# Transformações em Bases de Dados

Este notebook reúne os principais métodos do **pandas** para **transformar** dados depois que a etapa de limpeza (nulos, duplicatas, outliers) já foi feita.

Vamos cobrir, na ordem em que aparecem na aula:

1. **Aplicação de funções em colunas** — `.map()`, `.apply()`, `.applymap()`
2. **Transformações condicionais** — `np.where()`, `np.select()`, `pd.cut()`, `pd.qcut()`
3. **Junção e concatenação de datasets** — `pd.concat()`, `pd.merge()`, `.join()`

Cada seção tem uma célula de markdown explicando o conceito e, em seguida, células de código comentadas com exemplos práticos.

## 1. Aplicando funções a colunas

Transformar dados é o coração da análise. O pandas oferece diversos métodos para aplicar uma função a uma `Series` ou a um `DataFrame`, cada um com seu caso de uso ideal:

| Método | Onde atua | Uso ideal |
|---|---|---|
| `.map()` | Elemento a elemento, em **uma Series** | Substituir valores usando um dicionário (ou função simples) |
| `.apply()` | Ao longo de um **eixo** (coluna ou linha) | Lógica customizada mais complexa, inclusive usando várias colunas por linha |
| `.applymap()` (ou `.map()` no DataFrame, versões recentes) | Célula a célula, em **todo o DataFrame** | Limpeza/formatação global (ex: `strip()` em todas as colunas de texto) |

> ⚠️ `.applymap()` está **depreciado** nas versões mais recentes do pandas (a partir da 2.1). O substituto recomendado é `DataFrame.map()`, que faz exatamente a mesma coisa (aplicação célula a célula). Mostramos os dois abaixo para você reconhecer ambos.

Vamos criar uma base de vendas fictícia para os exemplos desta seção.

In [1]:
# Base de dados para os exemplos de .map(), .apply() e .applymap()
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "canal": ["Loja Física", "E-Commerce", "E-Commerce", "Loja Física", "E-Commerce",
              "Loja Física", "E-Commerce", "Loja Física", "E-Commerce", "Loja Física"],
    "regiao": [" SP ", "rj", " MG", "sp", "RJ ", " mg ", "SP", "rj ", " MG ", "sp "],
    "produto": ["Caneta", "Caderno", "Mochila", "Caneta", "Estojo",
                "Caderno", "Mochila", "Caneta", "Estojo", "Caderno"],
    "unidades_vendidas_cx": [80, 250, 420, 60, 310, 95, 180, 40, 260, 120],
    "receita_total": [1_500, 7_800, 13_200, 900, 9_600,
                       2_100, 5_400, 750, 8_300, 3_000],
})

df

,canal,regiao,produto,unidades_vendidas_cx,receita_total
0,Loja Física,SP,Caneta,80,1500
1,E-Commerce,rj,Caderno,250,7800
2,E-Commerce,MG,Mochila,420,13200
3,Loja Física,sp,Caneta,60,900
4,E-Commerce,RJ,Estojo,310,9600
5,Loja Física,mg,Caderno,95,2100
6,E-Commerce,SP,Mochila,180,5400
7,Loja Física,rj,Caneta,40,750
8,E-Commerce,MG,Estojo,260,8300
9,Loja Física,sp,Caderno,120,3000


### `.map()` — mapeamento elemento a elemento

**Conceito:** trabalha em **uma coluna só**. Você passa um dicionário (ou função) e ele substitui cada valor encontrado pelo correspondente.

**Contexto:** a coluna `canal` tem valores como `"Loja Física"` e `"E-Commerce"`. Queremos criar uma coluna nova com uma versão abreviada desses canais.

In [2]:
# Dicionário de mapeamento: valor original -> valor novo
mapa_canal = {
    "Loja Física": "Balcão",
    "E-Commerce": "Site",
}

# .map() aplica o dicionário elemento por elemento na Series
df["canal_abrev"] = df["canal"].map(mapa_canal)

print("=== .map() com dicionário ===")
df[["canal", "canal_abrev"]]

=== .map() com dicionário ===


,canal,canal_abrev
0,Loja Física,Balcão
1,E-Commerce,Site
2,E-Commerce,Site
3,Loja Física,Balcão
4,E-Commerce,Site
5,Loja Física,Balcão
6,E-Commerce,Site
7,Loja Física,Balcão
8,E-Commerce,Site
9,Loja Física,Balcão


In [3]:
# .map() também aceita uma FUNÇÃO em vez de um dicionário
# Exemplo: marcar se o canal é digital ou não
df["eh_digital"] = df["canal"].map(lambda c: c == "E-Commerce")

print("=== .map() com função (lambda) ===")
df[["canal", "eh_digital"]]

=== .map() com função (lambda) ===


,canal,eh_digital
0,Loja Física,False
1,E-Commerce,True
2,E-Commerce,True
3,Loja Física,False
4,E-Commerce,True
5,Loja Física,False
6,E-Commerce,True
7,Loja Física,False
8,E-Commerce,True
9,Loja Física,False


### `.apply()` — função ao longo de um eixo

**Conceito:** pode operar sobre **uma coluna inteira** (`axis=0`, o padrão quando aplicado numa Series) ou sobre **cada linha** do DataFrame (`axis=1`), recebendo a linha inteira como uma `Series`. É mais flexível que `.map()`: aceita funções customizadas mais complexas, inclusive combinando várias colunas.

**Contexto 1:** classificar cada venda em faixas de quantidade — Baixo, Médio ou Alto.

In [4]:
# Função que recebe UM valor (de uma coluna) e retorna a faixa correspondente
def classificar_valor(qtd):
    if qtd < 100:
        return "Baixo"
    elif qtd < 300:
        return "Médio"
    else:
        return "Alto"

# .apply() percorre cada elemento da coluna 'unidades_vendidas_cx' e aplica a função
df["faixa_unidades"] = df["unidades_vendidas_cx"].apply(classificar_valor)

print("=== .apply() em uma Series ===")
df[["unidades_vendidas_cx", "faixa_unidades"]]

=== .apply() em uma Series ===


,unidades_vendidas_cx,faixa_unidades
0,80,Baixo
1,250,Médio
2,420,Alto
3,60,Baixo
4,310,Alto
5,95,Baixo
6,180,Médio
7,40,Baixo
8,260,Médio
9,120,Médio


**Contexto 2:** agora um exemplo com `axis=1`, em que a função recebe a **linha inteira** e pode combinar várias colunas — por exemplo, calcular o ticket médio (receita dividida pelas unidades vendidas) de cada venda.

In [5]:
# Função que recebe uma LINHA inteira (Series) e retorna um valor calculado
def calcular_ticket_medio(linha):
    if linha["unidades_vendidas_cx"] == 0:
        return 0
    return round(linha["receita_total"] / linha["unidades_vendidas_cx"], 2)

# axis=1 -> a função é aplicada linha a linha, não coluna a coluna
df["ticket_medio"] = df.apply(calcular_ticket_medio, axis=1)

print("=== .apply() em um DataFrame, com axis=1 (linha a linha) ===")
df[["produto", "receita_total", "unidades_vendidas_cx", "ticket_medio"]]

=== .apply() em um DataFrame, com axis=1 (linha a linha) ===


,produto,receita_total,unidades_vendidas_cx,ticket_medio
0,Caneta,1500,80,18.75
1,Caderno,7800,250,31.20
2,Mochila,13200,420,31.43
3,Caneta,900,60,15.00
4,Estojo,9600,310,30.97
5,Caderno,2100,95,22.11
6,Mochila,5400,180,30.00
7,Caneta,750,40,18.75
8,Estojo,8300,260,31.92
9,Caderno,3000,120,25.00


### `.applymap()` / `DataFrame.map()` — célula por célula

**Conceito:** varre **todas as células** de um DataFrame (ou de um recorte dele). Ideal para limpeza e formatação global — por exemplo, remover espaços em branco que entraram na importação dos dados.

**Contexto:** a coluna `regiao` veio da fonte de dados com espaços extras e capitalização inconsistente (`" SP "`, `"rj"`, `" MG"`...). Queremos limpar todas as colunas de texto de uma vez.

In [6]:
# Função que limpa uma célula: se for string, tira espaços e deixa maiúscula; senão, devolve como está
def limpar_texto(celula):
    if isinstance(celula, str):
        return celula.strip().upper()
    return celula  # números e datas passam sem alteração

# Seleciona apenas as colunas de texto para aplicar a limpeza
colunas_texto = ["regiao", "canal", "produto"]

# Forma recomendada em pandas >= 2.1 (substitui o applymap, que está depreciado)
df[colunas_texto] = df[colunas_texto].map(limpar_texto)

# Caso esteja em uma versão mais antiga do pandas, o equivalente depreciado seria:
# df[colunas_texto] = df[colunas_texto].applymap(limpar_texto)

print("=== .map() no DataFrame inteiro (célula a célula) — equivalente ao .applymap() ===")
df[colunas_texto]

=== .map() no DataFrame inteiro (célula a célula) — equivalente ao .applymap() ===


,regiao,canal,produto
0,SP,LOJA FÍSICA,CANETA
1,RJ,E-COMMERCE,CADERNO
2,MG,E-COMMERCE,MOCHILA
3,SP,LOJA FÍSICA,CANETA
4,RJ,E-COMMERCE,ESTOJO
5,MG,LOJA FÍSICA,CADERNO
6,SP,E-COMMERCE,MOCHILA
7,RJ,LOJA FÍSICA,CANETA
8,MG,E-COMMERCE,ESTOJO
9,SP,LOJA FÍSICA,CADERNO


## 2. Transformações condicionais

Transformações condicionais permitem criar novas colunas com base em **regras de negócio**. No pandas, temos três abordagens principais, cada uma adequada para um nível diferente de complexidade lógica:

| Ferramenta | Lógica equivalente | Uso ideal |
|---|---|---|
| `np.where()` | `if / else` vetorizado | Uma condição → dois resultados possíveis |
| `np.select()` | `if / elif / elif ... / else` vetorizado | Múltiplas condições → múltiplos resultados |
| `pd.cut()` | Faixas com **limites definidos por você** | Segmentar valores contínuos em bins de negócio (ex: faixas salariais) |
| `pd.qcut()` | Faixas com **o mesmo número de registros** (quantis) | Criar rankings, percentis e segmentações equilibradas |

Vamos continuar usando o DataFrame `df` criado na seção anterior.

### `np.where()` — uma condição, dois resultados

**Conceito:** funciona como um `if/else` vetorizado (aplicado à coluna inteira de uma vez, sem loop).

**Lógica:** `np.where(condição, valor_se_verdadeiro, valor_se_falso)` — rápido, direto, e ideal quando a decisão é binária.

**Contexto:** classificar cada venda como `"Site"` ou `"Balcão"` com base na coluna `canal`.

In [7]:
# np.where(condição, valor se TRUE, valor se FALSE)
df["tipo_canal"] = np.where(
    df["canal"] == "E-Commerce",  # condição
    "Site",                       # se TRUE
    "Balcão"                      # se FALSE
)

print("=== np.where(): condição única ===")
df[["canal", "tipo_canal"]]

=== np.where(): condição única ===


,canal,tipo_canal
0,LOJA FÍSICA,Balcão
1,E-COMMERCE,Balcão
2,E-COMMERCE,Balcão
3,LOJA FÍSICA,Balcão
4,E-COMMERCE,Balcão
5,LOJA FÍSICA,Balcão
6,E-COMMERCE,Balcão
7,LOJA FÍSICA,Balcão
8,E-COMMERCE,Balcão
9,LOJA FÍSICA,Balcão


### `np.select()` — múltiplas condições

**Conceito:** o `elif` vetorizado. Você passa uma **lista de condições** e uma **lista de resultados** correspondentes. A primeira condição verdadeira "vence"; se nenhuma for verdadeira, aplica o `default`.

**Lógica:** `np.select([cond1, cond2, cond3], [res1, res2, res3], default=...)`

**Contexto:** classificar cada venda em uma categoria de desempenho com base na `receita_total`.

In [8]:
# Passo 1: definir as condições (em ordem de prioridade — a primeira que bater vence)
condicoes = [
    df["receita_total"] >= 10_000,                                     # faixa alta
    (df["receita_total"] >= 3_000) & (df["receita_total"] < 10_000),   # faixa média
    df["receita_total"] < 3_000,                                       # faixa baixa
]

# Passo 2: definir os resultados correspondentes a cada condição (mesma ordem/tamanho)
resultados = [
    "Alta Performance",
    "Média Performance",
    "Baixa Performance",
]

# Passo 3: aplicar o np.select()
df["desempenho"] = np.select(
    condicoes,
    resultados,
    default="Não classificado"  # fallback, caso nenhuma condição seja satisfeita
)

print("=== np.select(): múltiplas condições ===")
df[["receita_total", "desempenho"]]

=== np.select(): múltiplas condições ===


,receita_total,desempenho
0,1500,Baixa Performance
1,7800,Média Performance
2,13200,Alta Performance
3,900,Baixa Performance
4,9600,Média Performance
5,2100,Baixa Performance
6,5400,Média Performance
7,750,Baixa Performance
8,8300,Média Performance
9,3000,Média Performance


### `pd.cut()` — faixas de tamanho fixo (definidas por você)

**Conceito:** divide um intervalo de valores em *bins* com **limites definidos manualmente**. Ideal quando os critérios de negócio já são pré-definidos (ex: faixas salariais, categorias de ticket).

Por padrão, os intervalos são fechados à direita — `right=True` — ou seja, `(0, 3000]` inclui o 3000 mas não o 0.

**Contexto:** segmentar as vendas em faixas de receita com limites fixos de negócio.

In [9]:
# Definindo os limites das faixas manualmente
limites = [0, 2_000, 5_000, 9_000, 12_000, float("inf")]

# Definindo os rótulos para cada faixa (tem que ter um rótulo a menos que o número de limites)
rotulos = ["Muito Baixo", "Baixo", "Médio", "Alto", "Muito Alto"]

# pd.cut() faz o corte e atribui a faixa correspondente a cada valor
df["faixa_receita_cut"] = pd.cut(
    df["receita_total"],
    bins=limites,
    labels=rotulos,
    right=True,  # padrão: intervalo fechado à direita, ex: (2000, 5000]
)

print("=== pd.cut(): faixas com limites definidos manualmente ===")
print(df[["receita_total", "faixa_receita_cut"]])

print("\nDistribuição das faixas (pd.cut não garante quantidades iguais por faixa):")
print(df["faixa_receita_cut"].value_counts().sort_index())

=== pd.cut(): faixas com limites definidos manualmente ===
   receita_total faixa_receita_cut
0           1500       Muito Baixo
1           7800             Médio
2          13200        Muito Alto
3            900       Muito Baixo
4           9600              Alto
5           2100             Baixo
6           5400             Médio
7            750       Muito Baixo
8           8300             Médio
9           3000             Baixo

Distribuição das faixas (pd.cut não garante quantidades iguais por faixa):
faixa_receita_cut
Muito Baixo    3
Baixo          2
Médio          3
Alto           1
Muito Alto     1
Name: count, dtype: int64


### `pd.qcut()` — faixas por quantis (tamanhos equilibrados)

**Conceito:** divide os dados em bins com o **mesmo número de registros** (quantis). Você não controla os limites — o pandas os calcula automaticamente para equilibrar a distribuição. Ideal para criar rankings, percentis e segmentações justas.

**Contexto:** criar quartis de receita, dividindo as vendas em 4 grupos de tamanho igual (Q1 a Q4).

In [10]:
# q=4 -> quartis (aproximadamente 25% dos registros em cada grupo)
df["quartil_receita"] = pd.qcut(
    df["receita_total"],
    q=4,
    labels=["Q1 - Menor 25%", "Q2", "Q3", "Q4 - Maior 25%"],
)

print("=== pd.qcut(): faixas por quantis (quantidade equilibrada de registros) ===")
print(df[["receita_total", "quartil_receita"]])

print("\nDistribuição dos quartis (pd.qcut tende a equilibrar a quantidade por faixa):")
print(df["quartil_receita"].value_counts().sort_index())

=== pd.qcut(): faixas por quantis (quantidade equilibrada de registros) ===
   receita_total quartil_receita
0           1500  Q1 - Menor 25%
1           7800              Q3
2          13200  Q4 - Maior 25%
3            900  Q1 - Menor 25%
4           9600  Q4 - Maior 25%
5           2100              Q2
6           5400              Q3
7            750  Q1 - Menor 25%
8           8300  Q4 - Maior 25%
9           3000              Q2

Distribuição dos quartis (pd.qcut tende a equilibrar a quantidade por faixa):
quartil_receita
Q1 - Menor 25%    3
Q2                2
Q3                2
Q4 - Maior 25%    3
Name: count, dtype: int64


> **`pd.cut()` vs `pd.qcut()`:** `cut` fatia o **eixo de valores** em pedaços do tamanho que você escolher (podendo sobrar bins vazios ou desbalanceados); `qcut` fatia a **quantidade de registros**, ajustando os limites para que cada grupo tenha aproximadamente o mesmo número de linhas.

## 3. Unindo e concatenando datasets

Na prática, raramente trabalhamos com uma única tabela. Dados de vendas precisam ser cruzados com cadastros de clientes, metas regionais, catálogos de produtos e muito mais. Existem três ferramentas principais:

| Função | O que faz | Equivalente |
|---|---|---|
| `pd.concat()` | Empilha ou alinha DataFrames com a **mesma estrutura** (ex: unir arquivos mensais de vendas em um único DataFrame anual) | "Colar" planilhas de mesmo formato |
| `pd.merge()` | Combina tabelas **diferentes** com base em colunas-chave comuns | `JOIN` do SQL (suporta `inner`, `left`, `right`, `outer`) |
| `.join()` | Une DataFrames pelo **índice** | Atalho de conveniência quando as chaves já são o índice |

> **Diferença central:** `concat` não cruza nada, apenas empilha; `merge` cruza por **colunas**; `.join()` cruza pelo **índice** (mas aceita `how=` assim como o `merge`).

A seguir usamos uma base única, aplicando as variantes de cada método.

In [11]:
"""
Exemplos de pd.concat(), pd.merge() e .join()
Usando uma única base de dados, aplicando as variantes de cada método.
"""

import pandas as pd
import numpy as np

# =========================================================
# BASE DE DADOS ÚNICA
# =========================================================

# --- Vendas mensais (mesma estrutura, para os exemplos de concat) ---
vendas_jan = pd.DataFrame({
    "data": ["2024-01-05", "2024-01-20"],
    "produto": ["Caneta", "Caderno"],
    "valor": [50, 80],
})

vendas_fev = pd.DataFrame({
    "data": ["2024-02-03", "2024-02-18"],
    "produto": ["Caneta", "Mochila"],
    "valor": [45, 220],
})

vendas_mar = pd.DataFrame({
    "data": ["2024-03-10"],
    "produto": ["Caderno"],
    "valor": [90],
})

# --- Clientes e Pedidos (para os exemplos de merge e join) ---
clientes = pd.DataFrame({
    "id_cliente": [1, 2, 3],
    "nome": ["Ana", "Bruno", "Carla"],
})

pedidos = pd.DataFrame({
    "id_pedido": [101, 102, 103],
    "id_cliente": [1, 1, 4],   # cliente 4 não existe em `clientes`
    "valor": [200, 150, 90],
})

### `pd.concat()` — empilhar DataFrames com a mesma estrutura

Não cruza dados por uma chave, só junta os blocos, um atrás do outro (`axis=0`, padrão) ou lado a lado (`axis=1`).

In [12]:
# 1. pd.concat() — empilhar DataFrames com a mesma estrutura

# --- axis=0 (padrão): empilha LINHAS, uma base embaixo da outra ---
vendas_2024 = pd.concat(
    [vendas_jan, vendas_fev, vendas_mar],
    axis=0,
    ignore_index=True,  # evita índices repetidos (0,1,0,1,0...)
)

print("=== concat axis=0 (empilhar linhas) ===")
vendas_2024

=== concat axis=0 (empilhar linhas) ===


,data,produto,valor
0,2024-01-05,Caneta,50
1,2024-01-20,Caderno,80
2,2024-02-03,Caneta,45
3,2024-02-18,Mochila,220
4,2024-03-10,Caderno,90


`axis=1` empilha **colunas**: útil quando os DataFrames têm o mesmo número de linhas (mesmo índice) e representam colunas complementares da mesma observação.

In [13]:
# --- axis=1: empilha COLUNAS, uma base do lado da outra ---
# útil quando os DataFrames têm o mesmo número de linhas (mesmo índice)
# e representam colunas complementares da mesma observação

metas_jan = pd.DataFrame({"meta_valor": [60, 75]})  # df de 1 coluna, meta_valor, com valores 60 e 75

vendas_com_meta = pd.concat([vendas_jan[["produto", "valor"]], metas_jan], axis=1)
print("=== concat axis=1 (empilhar colunas) ===")
vendas_com_meta

=== concat axis=1 (empilhar colunas) ===


,produto,valor,meta_valor
0,Caneta,50,60
1,Caderno,80,75


Quando as bases têm **colunas diferentes**, `pd.concat()` alinha o que casa e preenche o restante com `NaN`.

In [14]:
# --- concat com colunas diferentes: gera NaN onde não bate ---
vendas_abr_extra = pd.DataFrame({
    "data": ["2024-04-01"],
    "produto": ["Estojo"],
    "valor": [35],
    "desconto": [0.1],  # coluna que não existe nas outras bases
})

vendas_com_gap = pd.concat([vendas_jan, vendas_abr_extra], ignore_index=True)
print("=== concat com colunas diferentes (gera NaN) ===")

vendas_com_gap

=== concat com colunas diferentes (gera NaN) ===


,data,produto,valor,desconto
0,2024-01-05,Caneta,50,NaN
1,2024-01-20,Caderno,80,NaN
2,2024-04-01,Estojo,35,0.1


### `pd.merge()` — cruzar tabelas diferentes por uma coluna-chave

Equivalente ao `JOIN` do SQL. O parâmetro `how=` controla o tipo de cruzamento:

- **`inner`** (padrão): só as linhas que existem nas **duas** tabelas.
- **`left`**: todas as linhas da tabela da esquerda, mesmo sem correspondência.
- **`right`**: todas as linhas da tabela da direita, mesmo sem correspondência.
- **`outer`**: união completa — todas as linhas de ambas, com `NaN` onde não há correspondência.

In [15]:
# 2. pd.merge() — cruzar tabelas diferentes por uma coluna-chave

# --- inner (padrão): só as linhas que existem nas DUAS tabelas ---
merge_inner = pd.merge(clientes, pedidos, on="id_cliente", how="inner")
print("=== merge how='inner' ===")
merge_inner

=== merge how='inner' ===


,id_cliente,nome,id_pedido,valor
0,1,Ana,101,200
1,1,Ana,102,150


In [16]:
# --- left: todas as linhas de `clientes`, mesmo sem pedido ---
merge_left = pd.merge(clientes, pedidos, on="id_cliente", how="left")
print("=== merge how='left' ===")
merge_left

=== merge how='left' ===


,id_cliente,nome,id_pedido,valor
0,1,Ana,101.0,200.0
1,1,Ana,102.0,150.0
2,2,Bruno,NaN,NaN
3,3,Carla,NaN,NaN


In [17]:
# --- right: todas as linhas de `pedidos`, mesmo sem cliente correspondente ---
merge_right = pd.merge(clientes, pedidos, on="id_cliente", how="right")
print("=== merge how='right' ===")
merge_right

=== merge how='right' ===


,id_cliente,nome,id_pedido,valor
0,1,Ana,101,200
1,1,Ana,102,150
2,4,NaN,103,90


In [18]:
# --- outer: união completa, com NaN onde não há correspondência ---
merge_outer = pd.merge(clientes, pedidos, on="id_cliente", how="outer")
print("=== merge how='outer' ===")
merge_outer

=== merge how='outer' ===


,id_cliente,nome,id_pedido,valor
0,1,Ana,101.0,200.0
1,1,Ana,102.0,150.0
2,2,Bruno,NaN,NaN
3,3,Carla,NaN,NaN
4,4,NaN,103.0,90.0


### `.join()` — mesma lógica do merge, mas cruzando pelo ÍNDICE

Serve para unir DataFrames pelo **índice**, não por uma coluna comum — é um atalho de conveniência para quando as chaves de junção já estão no índice de cada DataFrame. Por padrão funciona como um `left join`, mas também aceita `how=`.

In [19]:
# 3. .join() — mesma lógica do merge, mas cruzando pelo ÍNDICE

# preparar as mesmas bases, agora com id_cliente como ÍNDICE em vez de coluna
clientes_idx = clientes.set_index("id_cliente")
pedidos_idx = pedidos.set_index("id_cliente")

In [20]:
# --- join padrão: funciona como um LEFT join ---
join_default = clientes_idx.join(pedidos_idx, how="left")
print("=== join padrão (equivalente a how='left') ===")
join_default

=== join padrão (equivalente a how='left') ===


,nome,id_pedido,valor
id_cliente,,,
1,Ana,101.0,200.0
1,Ana,102.0,150.0
2,Bruno,NaN,NaN
3,Carla,NaN,NaN


In [21]:
# --- join também aceita how=, igual ao merge ---
join_inner = clientes_idx.join(pedidos_idx, how="inner")
print("=== join how='inner' ===")
join_inner

=== join how='inner' ===


,nome,id_pedido,valor
id_cliente,,,
1,Ana,101,200
1,Ana,102,150


In [22]:
join_outer = clientes_idx.join(pedidos_idx, how="outer")
print("=== join how='outer' ===")
join_outer

=== join how='outer' ===


,nome,id_pedido,valor
id_cliente,,,
1,Ana,101.0,200.0
1,Ana,102.0,150.0
2,Bruno,NaN,NaN
3,Carla,NaN,NaN
4,NaN,103.0,90.0


Para reforçar a equivalência entre os dois: dá pra fazer o mesmo cruzamento por índice usando `pd.merge()` diretamente, com `left_index=True, right_index=True`.

In [23]:
# --- comparação: merge equivalente ao join acima, usando left_index/right_index ---
merge_equivalente_ao_join = pd.merge(
    clientes_idx, pedidos_idx,
    left_index=True, right_index=True,
    how="left",
)
print("=== merge com left_index=True, right_index=True (equivalente ao .join) ===")
merge_equivalente_ao_join

=== merge com left_index=True, right_index=True (equivalente ao .join) ===


,nome,id_pedido,valor
id_cliente,,,
1,Ana,101.0,200.0
1,Ana,102.0,150.0
2,Bruno,NaN,NaN
3,Carla,NaN,NaN


## Resumindo

| Situação | Função recomendada |
|---|---|
| Substituir valores de uma coluna por um dicionário simples | `.map()` |
| Aplicar lógica customizada em uma coluna ou combinando colunas de uma linha | `.apply()` |
| Limpar/formatar todas as células de um DataFrame de uma vez | `.applymap()` / `DataFrame.map()` |
| Uma condição, dois resultados possíveis | `np.where()` |
| Várias condições, vários resultados possíveis | `np.select()` |
| Segmentar valores em faixas com limites definidos por regra de negócio | `pd.cut()` |
| Segmentar valores em faixas com a mesma quantidade de registros (quantis) | `pd.qcut()` |
| Juntar arquivos com a mesma estrutura (ex: meses de vendas) | `pd.concat()` |
| Cruzar duas tabelas por uma coluna em comum (ex: `id`), com controle de inner/left/right/outer | `pd.merge()` |
| Cruzar duas tabelas que já usam a mesma coluna como índice | `.join()` |